## SVM

In [ ]:
X1, X2, X3, X4, X5 = create_kfold_sets(data_train=data_train) #Model without feature selection
#X1, X2, X3, X4, X5 = create_kfold_sets(data_train=data_train_wrapper) #Model with feature selection

D = [X1, X2, X3, X4, X5]

In [ ]:
# Define the columns for the results DataFrame
columns = ['Model', 'C', 'Kernel', 'Accuracy', 'Recall', 'Specificity', 'Precision', 'F1']
df_results = pd.DataFrame(columns=columns)

# Hyperparameter values for SVM
C_values = [0.001, 0.01, 0.1, 1]  # Regularization parameter values
kernels = ['linear', 'rbf']  # Kernel types to test
model_name = 'SVM'  # Name of the model

# Iterate over each combination of C and kernel
for c in C_values:
    for kernel in kernels:
        df_results_fold = pd.DataFrame(columns=['Model', 'Accuracy', 'Recall', 'Specificity', 'Precision', 'F1'])
        # Perform 5-fold cross-validation
        for fold in range(5):
            # Split data into training and testing sets for this fold
            test_data = pd.concat([D[fold]])
            y_test = test_data['label_binary']
            X_test = test_data.drop(columns=['label_binary', 'n_image', 'label_multi'])

            train_data = pd.concat([D[i] for i in range(5) if i != fold], ignore_index=True)
            y_train = train_data['label_binary']
            X_train = train_data.drop(columns=['label_binary', 'n_image', 'label_multi'])

            # Scale the features using StandardScaler
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            # Train the SVM model with the current hyperparameters
            model = SVC(C=c, kernel=kernel, verbose=False)
            model.fit(X_train_scaled, y_train)

            y_pred = model.predict(X_test_scaled)

            # Print model details and confusion matrix
            print(f'Model with C={c} and kernel={kernel}:')
            labels = ('corrosion', 'no corrosion')
            results= evaluate_model(y_pred, y_test, model=model_name, labels=labels)
            df_results_fold = pd.concat([df_results_fold, results], ignore_index=True)



        mean_accuracy, mean_recall, mean_specificity, mean_precision, mean_f1= means_results(df_results_fold)

        # Create a dictionary to store the results for this configuration
        result_row = {
            'Model': 'SVM',
            'C': c,
            'Kernel': kernel,
            'Accuracy': mean_accuracy,
            'Recall': mean_recall,
            'Specificity': mean_specificity,
            'Precision': mean_precision,
            'F1': mean_f1,
        }

        # Append the results to the DataFrame
        df_results = pd.concat([df_results, pd.DataFrame([result_row])], ignore_index=True)

# Display final results summary
print("\nFinal Results:")
print(df_results.round(3))

In [ ]:
C_VALUE = 0.1                # Regularization parameter
KERNEL_TYPE = 'rbf'          # Kernel type for SVM
LABEL_MAPPING = {            # Label encoding dictionary
    'no corrosion': 0,
    'corrosion': 1
}
model_name = 'SVM'           # Name of the model

# ========================
# Data Preparation
# ========================
# Separate features and labels for training data
X_train = data_train.drop(columns=['label_binary', 'label_multi', 'n_image'])
y_train = data_train['label_binary']

# Separate features and labels for test data
X_test = data_test.drop(columns=['label_binary', 'label_multi', 'n_image'])
y_test = data_test['label_binary']

# ========================
# Feature Scaling
# ========================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ========================
# Label Encoding
# ========================
y_train_encoded = y_train.map(LABEL_MAPPING)
y_test_encoded = y_test.map(LABEL_MAPPING)

# ========================
# Model Training
# ========================
model = SVC(C=C_VALUE, kernel=KERNEL_TYPE, verbose=True)
model.fit(X_train_scaled, y_train_encoded)

# ========================
# Model Evaluation
# ========================
# Time prediction only
start_time = time.time()
y_pred = model.predict(X_test_scaled)
execution_time = round(time.time() - start_time, 3)

recall, specificity, precision, f1, accuracy = evaluate_model(y_pred, y_test_encoded, model=model_name, labels=(1,0))
print(f'Execution Time: {execution_time} seconds')

# ========================
# Results Storage
# ========================
results_columns = ['Model', 'C', 'Kernel', 'Accuracy',
                  'Recall', 'Specificity', 'Precision', 'F1', 'Time']
results_data = {
    'Model': 'SVM',
    'C': C_VALUE,
    'Kernel': KERNEL_TYPE,
    'Accuracy': accuracy,
    'Recall': recall,
    'Specificity': specificity,
    'Precision': precision,
    'F1': f1,
    'Time': execution_time
}

df_results = pd.DataFrame([results_data])
print('\nResults DataFrame:')
print(df_results)
